In [2]:
import os
import random
import json
from collections import defaultdict
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense
from mravens_icons import *


# -------------------------------
# Config
# -------------------------------
with open("C:/Users/somas/Paper_exp/Enc_Dec_v7/config.json", "r") as config_file:
    Config = json.load(config_file)

Config["file"] = "C:/Users/somas/Paper_exp/Enc_Dec_v7/Mem_data/3T1R_New.xlsx"
Config["Res_Column"] = 'Read Res.'
Config["Current_Column"] = 'Read_Current (µA)'
Config["Power_Column"] = 'Read_Power (µW)'

# -------------------------------
# Helper Functions
# -------------------------------
def one_hot(y, n=None):
    if n is None:
        n = np.max(y) + 1
    y_onehot = np.zeros((len(y), n))
    for i, j in enumerate(y):
        y_onehot[i, j] = 1
    return y_onehot


def rate_encode(X_rate, f_sample):
    """
    Converts integer-valued rate features into spike trains.

    X_rate shape: (m, f)
    Output shape: (m, f, f_sample)
    """
    X_rate_data = np.zeros((X_rate.shape[0], X_rate.shape[1], f_sample), dtype=np.float32)

    for m, sample in enumerate(X_rate):
        for n, feature in enumerate(sample):
            if feature > 0:
                step = max(1, round(f_sample / feature))
                X_rate_data[m, n, 0:-1:step] = 1

    return X_rate_data


def simulate(X_rate_data, keras_weights, f_sample, st_point, n_id, type_id, abs_period):
    """
    Neuromorphic simulation using MRAVENS.
    Kept here from your original code for completeness.
    """
    y_pred = []
    spikes_node = []

    for i in range(len(X_rate_data)):
        net = create_custom_network()
        net.add_stimuli(st_point, n_id, X_rate_data[i], keras_weights)

        for (nid, t, p) in zip(n_id, type_id, abs_period):
            net.add_neuron(nid, t)
            net.neuron[nid].abs_period = p

        proc = MRAVENS(-5, 5)
        event = process_event(f_sample, net, proc)
        event.apply_spike()

        spikes = np.sum(event.spikes_, axis=0)
        y_pred.append(np.argmax(spikes))
        spikes_node.append(spikes)

    return np.array(spikes_node), np.array(y_pred)


def weight_to_Resistance(weights, R_max, R_min):
    """
    Map trained weights to resistance values using linear conductance scaling.
    weights should be model.get_weights()
    """
    w = weights[0]   # shape: (f, c)

    G_H = 1 / (R_min * 1e3)
    G_L = 1 / (R_max * 1e3)

    mu = np.min(w)
    W_H = np.max(w)

    alpha = (G_H - G_L) / (W_H + abs(mu))
    G = alpha * (w + abs(mu)) + G_L

    R = tf.constant(((G ** -1) * 1e-3).astype(int))
    return R


def table_creation(config):
    """
    Reads lookup table from Excel and creates a TensorFlow hash table.
    Uses Power column for power estimation.
    """
    file = config["file"]
    res_col = config["Res_Column"]
    power_col = config["Power_Column"]

    df = pd.read_excel(file)

    R_table = df[res_col].values.astype(int)
    P_table = df[power_col].values.astype(np.float32)

    R_min, R_max = R_table[0], R_table[-1]

    keys_tensor = tf.constant(R_table)
    vals_tensor = tf.constant(P_table)

    init = tf.lookup.KeyValueTensorInitializer(keys_tensor, vals_tensor)
    table = tf.lookup.StaticHashTable(init, default_value=-1.0)

    return table, R_min, R_max


def set_seed(seed_value):
    os.environ["PYTHONHASHSEED"] = str(seed_value)
    random.seed(seed_value)
    np.random.seed(seed_value)
    tf.random.set_seed(seed_value)


def prepare_dataset(seed_value, f_sample):
    """
    Load dataset, normalize, split, and rate encode.
    """
    data = load_breast_cancer()
    
    X, y = data["data"], data["target"]

    n_bins = np.max(y) + 1
    y_onehot = one_hot(y, n_bins)

    X_norm = (X - np.min(X, axis=0)) / (np.max(X, axis=0) - np.min(X, axis=0))

    X_train, X_test, y_train, y_test = train_test_split(
        X_norm,
        y_onehot,
        test_size=0.30,
        random_state=seed_value,
        stratify=y
    )

    X_rate_train = (X_train * f_sample).astype(int)
    X_rate_test = (X_test * f_sample).astype(int)

    X_rate_data_train = rate_encode(X_rate_train, f_sample)
    X_rate_data_test = rate_encode(X_rate_test, f_sample)

    return X_train, X_test, y_train, y_test, X_rate_data_train, X_rate_data_test, X_norm.shape[1], y_onehot.shape[1]


def build_and_train_model(input_dim, output_dim, seed_value, X_train, y_train, X_test, y_test):
    """
    Train a single-layer TF model.
    """
    initializer = tf.keras.initializers.HeNormal(seed=seed_value)

    model = Sequential([
        Dense(
            units=output_dim,
            input_shape=(input_dim,),
            activation="sigmoid",
            use_bias=False,
            kernel_initializer=initializer
        )
    ])

    opt = tf.keras.optimizers.Adam(
        learning_rate=0.05,
        beta_1=0.7,
        beta_2=0.999
    )

    model.compile(
        loss="categorical_crossentropy",
        optimizer=opt,
        metrics=["accuracy"]
    )

    history = model.fit(
        X_train,
        y_train,
        epochs=100,
        batch_size=16,
        verbose=0,
        validation_data=(X_test, y_test),
        shuffle=True
    )

    train_loss, train_acc = model.evaluate(X_train, y_train, verbose=0)
    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)

    return model, history, train_loss, train_acc, test_loss, test_acc


def compute_mean_power(X_rate_data, power_matrix):
    """
    Compute power tensor over all time steps.

    X_rate_data shape: (m, f, T)
    power_matrix shape: (f, c)

    Output shape: (m, c, T)
    """
    T = X_rate_data.shape[2]

    Y_power = np.stack([np.sum(np.dot(X_rate_data_train[:,:,t], power_matrix), axis=1) for t in range(T)], axis=1)

    return np.mean(np.mean(Y_power,axis=1)), Y_power


# -------------------------------
# Main
# -------------------------------
f_samples = [8, 16, 32, 64]
# f_samples = [8]
seeds = [1, 8, 10, 11, 21, 25, 28, 32, 36]
# seeds = [1]

power_results = defaultdict(list)
summary_rows = []

table, R_min, R_max = table_creation(Config)

for f_sample in f_samples:
    print(f"\n{'='*50}")
    print(f"f_sample = {f_sample}")
    print(f"{'='*50}")

    for seed_value in seeds:
        set_seed(seed_value)

        X_train, X_test, y_train, y_test, X_rate_data_train, X_rate_data_test, input_dim, output_dim = prepare_dataset(
            seed_value, f_sample
        )

        model, history, train_loss, train_acc, test_loss, test_acc = build_and_train_model(
            input_dim, output_dim, seed_value, X_train, y_train, X_test, y_test
        )

        R = weight_to_Resistance(model.get_weights(), R_max, R_min)

        # shape: (f, c)
        power_matrix = table.lookup(R).numpy()

        mean_power_train, Y_power_train = compute_mean_power(
            X_rate_data_train,
            power_matrix
        )

        power_results[f_sample].append(mean_power_train)

        summary_rows.append({
            "f_sample": f_sample,
            "seed": seed_value,
            "train_acc": train_acc,
            "test_acc": test_acc,
            "mean_power_train": mean_power_train
        })

        print(
            f"Seed {seed_value:2d} | "
            f"Train Acc = {train_acc:.4f} | "
            f"Test Acc = {test_acc:.4f} | "
            f"Mean Power = {mean_power_train:.6e}"
        )

    mean_over_seeds = np.mean(power_results[f_sample])
    std_over_seeds = np.std(power_results[f_sample])

    print(f"\nMean power over all seeds for f_sample={f_sample}: {mean_over_seeds:.6e}")
    print(f"Std  power over all seeds for f_sample={f_sample}: {std_over_seeds:.6e}")

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv("power_summary_by_seed.csv", index=False)

mean_df = (
    summary_df.groupby("f_sample", as_index=False)["mean_power_train"]
    .agg(["mean", "std"])
    .reset_index()
)
mean_df.to_csv("power_summary_by_fsample.csv", index=False)

print("\nSaved:")
print(" - power_summary_by_seed.csv")
print(" - power_summary_by_fsample.csv")

print("\nFinal mean power by f_sample:")
print(mean_df)


f_sample = 8
Seed  1 | Train Acc = 0.9724 | Test Acc = 0.9240 | Mean Power = 3.822165e+01
Seed  8 | Train Acc = 0.9648 | Test Acc = 0.9532 | Mean Power = 3.817419e+01
Seed 10 | Train Acc = 0.9648 | Test Acc = 0.9474 | Mean Power = 3.824541e+01
Seed 11 | Train Acc = 0.9623 | Test Acc = 0.9591 | Mean Power = 3.826687e+01
Seed 21 | Train Acc = 0.9623 | Test Acc = 0.9708 | Mean Power = 3.781973e+01
Seed 25 | Train Acc = 0.9623 | Test Acc = 0.9708 | Mean Power = 3.744166e+01
Seed 28 | Train Acc = 0.9724 | Test Acc = 0.9591 | Mean Power = 3.759096e+01
Seed 32 | Train Acc = 0.9673 | Test Acc = 0.9474 | Mean Power = 3.793214e+01
Seed 36 | Train Acc = 0.9774 | Test Acc = 0.9474 | Mean Power = 3.780302e+01

Mean power over all seeds for f_sample=8: 3.794395e+01
Std  power over all seeds for f_sample=8: 2.862815e-01

f_sample = 16
Seed  1 | Train Acc = 0.9724 | Test Acc = 0.9240 | Mean Power = 4.467629e+01
Seed  8 | Train Acc = 0.9648 | Test Acc = 0.9532 | Mean Power = 4.475079e+01
Seed 10 | Tra

In [3]:
import os
import random
import json
from collections import defaultdict
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense
from mravens_icons import *


# -------------------------------
# Config
# -------------------------------
with open("C:/Users/somas/Paper_exp/Enc_Dec_v7/config.json", "r") as config_file:
    Config = json.load(config_file)

Config["file"] = "C:/Users/somas/Paper_exp/Enc_Dec_v7/Mem_data/3T1R_New.xlsx"
Config["Res_Column"] = 'Read Res.'
Config["Current_Column"] = 'Read_Current (µA)'
Config["Power_Column"] = 'Read_Power (µW)'

# -------------------------------
# Helper Functions
# -------------------------------
def one_hot(y, n=None):
    if n is None:
        n = np.max(y) + 1
    y_onehot = np.zeros((len(y), n))
    for i, j in enumerate(y):
        y_onehot[i, j] = 1
    return y_onehot


def rate_encode(X_rate, f_sample):
    """
    Converts integer-valued rate features into spike trains.

    X_rate shape: (m, f)
    Output shape: (m, f, f_sample)
    """
    X_rate_data = np.zeros((X_rate.shape[0], X_rate.shape[1], f_sample), dtype=np.float32)

    for m, sample in enumerate(X_rate):
        for n, feature in enumerate(sample):
            if feature > 0:
                step = max(1, round(f_sample / feature))
                X_rate_data[m, n, 0:-1:step] = 1

    return X_rate_data


def simulate(X_rate_data, keras_weights, f_sample, st_point, n_id, type_id, abs_period):
    """
    Neuromorphic simulation using MRAVENS.
    Kept here from your original code for completeness.
    """
    y_pred = []
    spikes_node = []

    for i in range(len(X_rate_data)):
        net = create_custom_network()
        net.add_stimuli(st_point, n_id, X_rate_data[i], keras_weights)

        for (nid, t, p) in zip(n_id, type_id, abs_period):
            net.add_neuron(nid, t)
            net.neuron[nid].abs_period = p

        proc = MRAVENS(-5, 5)
        event = process_event(f_sample, net, proc)
        event.apply_spike()

        spikes = np.sum(event.spikes_, axis=0)
        y_pred.append(np.argmax(spikes))
        spikes_node.append(spikes)

    return np.array(spikes_node), np.array(y_pred)


def weight_to_Resistance(weights, R_max, R_min):
    """
    Map trained weights to resistance values using linear conductance scaling.
    weights should be model.get_weights()
    """
    w = weights[0]   # shape: (f, c)

    G_H = 1 / (R_min * 1e3)
    G_L = 1 / (R_max * 1e3)

    mu = np.min(w)
    W_H = np.max(w)

    alpha = (G_H - G_L) / (W_H + abs(mu))
    G = alpha * (w + abs(mu)) + G_L

    R = tf.constant(((G ** -1) * 1e-3).astype(int))
    return R


def table_creation(config):
    """
    Reads lookup table from Excel and creates a TensorFlow hash table.
    Uses Power column for power estimation.
    """
    file = config["file"]
    res_col = config["Res_Column"]
    power_col = config["Power_Column"]

    df = pd.read_excel(file)

    R_table = df[res_col].values.astype(int)
    P_table = df[power_col].values.astype(np.float32)

    R_min, R_max = R_table[0], R_table[-1]

    keys_tensor = tf.constant(R_table)
    vals_tensor = tf.constant(P_table)

    init = tf.lookup.KeyValueTensorInitializer(keys_tensor, vals_tensor)
    table = tf.lookup.StaticHashTable(init, default_value=-1.0)

    return table, R_min, R_max


def set_seed(seed_value):
    os.environ["PYTHONHASHSEED"] = str(seed_value)
    random.seed(seed_value)
    np.random.seed(seed_value)
    tf.random.set_seed(seed_value)


def prepare_dataset(seed_value, f_sample):
    """
    Load dataset, normalize, split, and rate encode.
    """
    data = load_iris()
    
    X, y = data["data"], data["target"]

    n_bins = np.max(y) + 1
    y_onehot = one_hot(y, n_bins)

    X_norm = (X - np.min(X, axis=0)) / (np.max(X, axis=0) - np.min(X, axis=0))

    X_train, X_test, y_train, y_test = train_test_split(
        X_norm,
        y_onehot,
        test_size=0.30,
        random_state=seed_value,
        stratify=y
    )

    X_rate_train = (X_train * f_sample).astype(int)
    X_rate_test = (X_test * f_sample).astype(int)

    X_rate_data_train = rate_encode(X_rate_train, f_sample)
    X_rate_data_test = rate_encode(X_rate_test, f_sample)

    return X_train, X_test, y_train, y_test, X_rate_data_train, X_rate_data_test, X_norm.shape[1], y_onehot.shape[1]


def build_and_train_model(input_dim, output_dim, seed_value, X_train, y_train, X_test, y_test):
    """
    Train a single-layer TF model.
    """
    initializer = tf.keras.initializers.HeNormal(seed=seed_value)

    model = Sequential([
        Dense(
            units=output_dim,
            input_shape=(input_dim,),
            activation="sigmoid",
            use_bias=False,
            kernel_initializer=initializer
        )
    ])

    opt = tf.keras.optimizers.Adam(
        learning_rate=0.05,
        beta_1=0.7,
        beta_2=0.999
    )

    model.compile(
        loss="categorical_crossentropy",
        optimizer=opt,
        metrics=["accuracy"]
    )

    history = model.fit(
        X_train,
        y_train,
        epochs=100,
        batch_size=16,
        verbose=0,
        validation_data=(X_test, y_test),
        shuffle=True
    )

    train_loss, train_acc = model.evaluate(X_train, y_train, verbose=0)
    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)

    return model, history, train_loss, train_acc, test_loss, test_acc


def compute_mean_power(X_rate_data, power_matrix):
    """
    Compute power tensor over all time steps.

    X_rate_data shape: (m, f, T)
    power_matrix shape: (f, c)

    Output shape: (m, c, T)
    """
    T = X_rate_data.shape[2]

    Y_power = np.stack([np.sum(np.dot(X_rate_data_train[:,:,t], power_matrix), axis=1) for t in range(T)], axis=1)

    return np.mean(np.mean(Y_power,axis=1)), Y_power


# -------------------------------
# Main
# -------------------------------
f_samples = [8, 16, 32, 64]
# f_samples = [8]
seeds = [1, 8, 10, 11, 21, 25, 28, 32, 36]
# seeds = [1]

power_results = defaultdict(list)
summary_rows = []

table, R_min, R_max = table_creation(Config)

for f_sample in f_samples:
    print(f"\n{'='*50}")
    print(f"f_sample = {f_sample}")
    print(f"{'='*50}")

    for seed_value in seeds:
        set_seed(seed_value)

        X_train, X_test, y_train, y_test, X_rate_data_train, X_rate_data_test, input_dim, output_dim = prepare_dataset(
            seed_value, f_sample
        )

        model, history, train_loss, train_acc, test_loss, test_acc = build_and_train_model(
            input_dim, output_dim, seed_value, X_train, y_train, X_test, y_test
        )

        R = weight_to_Resistance(model.get_weights(), R_max, R_min)

        # shape: (f, c)
        power_matrix = table.lookup(R).numpy()

        mean_power_train, Y_power_train = compute_mean_power(
            X_rate_data_train,
            power_matrix
        )

        power_results[f_sample].append(mean_power_train)

        summary_rows.append({
            "f_sample": f_sample,
            "seed": seed_value,
            "train_acc": train_acc,
            "test_acc": test_acc,
            "mean_power_train": mean_power_train
        })

        print(
            f"Seed {seed_value:2d} | "
            f"Train Acc = {train_acc:.4f} | "
            f"Test Acc = {test_acc:.4f} | "
            f"Mean Power = {mean_power_train:.6e}"
        )

    mean_over_seeds = np.mean(power_results[f_sample])
    std_over_seeds = np.std(power_results[f_sample])

    print(f"\nMean power over all seeds for f_sample={f_sample}: {mean_over_seeds:.6e}")
    print(f"Std  power over all seeds for f_sample={f_sample}: {std_over_seeds:.6e}")

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv("power_summary_by_seed.csv", index=False)

mean_df = (
    summary_df.groupby("f_sample", as_index=False)["mean_power_train"]
    .agg(["mean", "std"])
    .reset_index()
)
mean_df.to_csv("power_summary_by_fsample.csv", index=False)

print("\nSaved:")
print(" - power_summary_by_seed.csv")
print(" - power_summary_by_fsample.csv")

print("\nFinal mean power by f_sample:")
print(mean_df)


f_sample = 8
Seed  1 | Train Acc = 0.7810 | Test Acc = 0.7556 | Mean Power = 1.643182e+01
Seed  8 | Train Acc = 0.7619 | Test Acc = 0.7111 | Mean Power = 1.636628e+01
Seed 10 | Train Acc = 0.8000 | Test Acc = 0.7333 | Mean Power = 1.620579e+01
Seed 11 | Train Acc = 0.8190 | Test Acc = 0.8000 | Mean Power = 1.637406e+01
Seed 21 | Train Acc = 0.7714 | Test Acc = 0.8000 | Mean Power = 1.608376e+01
Seed 25 | Train Acc = 0.8667 | Test Acc = 0.7111 | Mean Power = 1.627872e+01
Seed 28 | Train Acc = 0.7905 | Test Acc = 0.7556 | Mean Power = 1.613916e+01
Seed 32 | Train Acc = 0.8000 | Test Acc = 0.6889 | Mean Power = 1.576823e+01
Seed 36 | Train Acc = 0.8000 | Test Acc = 0.8000 | Mean Power = 1.591025e+01

Mean power over all seeds for f_sample=8: 1.617312e+01
Std  power over all seeds for f_sample=8: 2.105573e-01

f_sample = 16
Seed  1 | Train Acc = 0.7810 | Test Acc = 0.7556 | Mean Power = 1.873651e+01
Seed  8 | Train Acc = 0.7619 | Test Acc = 0.7111 | Mean Power = 1.869435e+01
Seed 10 | Tra

In [4]:
import os
import random
import json
from collections import defaultdict
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense
from mravens_icons import *


# -------------------------------
# Config
# -------------------------------
with open("C:/Users/somas/Paper_exp/Enc_Dec_v7/config.json", "r") as config_file:
    Config = json.load(config_file)

Config["file"] = "C:/Users/somas/Paper_exp/Enc_Dec_v7/Mem_data/3T1R_New.xlsx"
Config["Res_Column"] = 'Read Res.'
Config["Current_Column"] = 'Read_Current (µA)'
Config["Power_Column"] = 'Read_Power (µW)'

# -------------------------------
# Helper Functions
# -------------------------------
def one_hot(y, n=None):
    if n is None:
        n = np.max(y) + 1
    y_onehot = np.zeros((len(y), n))
    for i, j in enumerate(y):
        y_onehot[i, j] = 1
    return y_onehot


def rate_encode(X_rate, f_sample):
    """
    Converts integer-valued rate features into spike trains.

    X_rate shape: (m, f)
    Output shape: (m, f, f_sample)
    """
    X_rate_data = np.zeros((X_rate.shape[0], X_rate.shape[1], f_sample), dtype=np.float32)

    for m, sample in enumerate(X_rate):
        for n, feature in enumerate(sample):
            if feature > 0:
                step = max(1, round(f_sample / feature))
                X_rate_data[m, n, 0:-1:step] = 1

    return X_rate_data


def simulate(X_rate_data, keras_weights, f_sample, st_point, n_id, type_id, abs_period):
    """
    Neuromorphic simulation using MRAVENS.
    Kept here from your original code for completeness.
    """
    y_pred = []
    spikes_node = []

    for i in range(len(X_rate_data)):
        net = create_custom_network()
        net.add_stimuli(st_point, n_id, X_rate_data[i], keras_weights)

        for (nid, t, p) in zip(n_id, type_id, abs_period):
            net.add_neuron(nid, t)
            net.neuron[nid].abs_period = p

        proc = MRAVENS(-5, 5)
        event = process_event(f_sample, net, proc)
        event.apply_spike()

        spikes = np.sum(event.spikes_, axis=0)
        y_pred.append(np.argmax(spikes))
        spikes_node.append(spikes)

    return np.array(spikes_node), np.array(y_pred)


def weight_to_Resistance(weights, R_max, R_min):
    """
    Map trained weights to resistance values using linear conductance scaling.
    weights should be model.get_weights()
    """
    w = weights[0]   # shape: (f, c)

    G_H = 1 / (R_min * 1e3)
    G_L = 1 / (R_max * 1e3)

    mu = np.min(w)
    W_H = np.max(w)

    alpha = (G_H - G_L) / (W_H + abs(mu))
    G = alpha * (w + abs(mu)) + G_L

    R = tf.constant(((G ** -1) * 1e-3).astype(int))
    return R


def table_creation(config):
    """
    Reads lookup table from Excel and creates a TensorFlow hash table.
    Uses Power column for power estimation.
    """
    file = config["file"]
    res_col = config["Res_Column"]
    power_col = config["Power_Column"]

    df = pd.read_excel(file)

    R_table = df[res_col].values.astype(int)
    P_table = df[power_col].values.astype(np.float32)

    R_min, R_max = R_table[0], R_table[-1]

    keys_tensor = tf.constant(R_table)
    vals_tensor = tf.constant(P_table)

    init = tf.lookup.KeyValueTensorInitializer(keys_tensor, vals_tensor)
    table = tf.lookup.StaticHashTable(init, default_value=-1.0)

    return table, R_min, R_max


def set_seed(seed_value):
    os.environ["PYTHONHASHSEED"] = str(seed_value)
    random.seed(seed_value)
    np.random.seed(seed_value)
    tf.random.set_seed(seed_value)


def prepare_dataset(seed_value, f_sample):
    """
    Load dataset, normalize, split, and rate encode.
    """
    data = load_wine()
    
    X, y = data["data"], data["target"]

    n_bins = np.max(y) + 1
    y_onehot = one_hot(y, n_bins)

    X_norm = (X - np.min(X, axis=0)) / (np.max(X, axis=0) - np.min(X, axis=0))

    X_train, X_test, y_train, y_test = train_test_split(
        X_norm,
        y_onehot,
        test_size=0.30,
        random_state=seed_value,
        stratify=y
    )

    X_rate_train = (X_train * f_sample).astype(int)
    X_rate_test = (X_test * f_sample).astype(int)

    X_rate_data_train = rate_encode(X_rate_train, f_sample)
    X_rate_data_test = rate_encode(X_rate_test, f_sample)

    return X_train, X_test, y_train, y_test, X_rate_data_train, X_rate_data_test, X_norm.shape[1], y_onehot.shape[1]


def build_and_train_model(input_dim, output_dim, seed_value, X_train, y_train, X_test, y_test):
    """
    Train a single-layer TF model.
    """
    initializer = tf.keras.initializers.HeNormal(seed=seed_value)

    model = Sequential([
        Dense(
            units=output_dim,
            input_shape=(input_dim,),
            activation="sigmoid",
            use_bias=False,
            kernel_initializer=initializer
        )
    ])

    opt = tf.keras.optimizers.Adam(
        learning_rate=0.05,
        beta_1=0.7,
        beta_2=0.999
    )

    model.compile(
        loss="categorical_crossentropy",
        optimizer=opt,
        metrics=["accuracy"]
    )

    history = model.fit(
        X_train,
        y_train,
        epochs=100,
        batch_size=16,
        verbose=0,
        validation_data=(X_test, y_test),
        shuffle=True
    )

    train_loss, train_acc = model.evaluate(X_train, y_train, verbose=0)
    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)

    return model, history, train_loss, train_acc, test_loss, test_acc


def compute_mean_power(X_rate_data, power_matrix):
    """
    Compute power tensor over all time steps.

    X_rate_data shape: (m, f, T)
    power_matrix shape: (f, c)

    Output shape: (m, c, T)
    """
    T = X_rate_data.shape[2]

    Y_power = np.stack([np.sum(np.dot(X_rate_data_train[:,:,t], power_matrix), axis=1) for t in range(T)], axis=1)

    return np.mean(np.mean(Y_power,axis=1)), Y_power


# -------------------------------
# Main
# -------------------------------
f_samples = [8, 16, 32, 64]
# f_samples = [8]
seeds = [1, 8, 10, 11, 21, 25, 28, 32, 36]
# seeds = [1]

power_results = defaultdict(list)
summary_rows = []

table, R_min, R_max = table_creation(Config)

for f_sample in f_samples:
    print(f"\n{'='*50}")
    print(f"f_sample = {f_sample}")
    print(f"{'='*50}")

    for seed_value in seeds:
        set_seed(seed_value)

        X_train, X_test, y_train, y_test, X_rate_data_train, X_rate_data_test, input_dim, output_dim = prepare_dataset(
            seed_value, f_sample
        )

        model, history, train_loss, train_acc, test_loss, test_acc = build_and_train_model(
            input_dim, output_dim, seed_value, X_train, y_train, X_test, y_test
        )

        R = weight_to_Resistance(model.get_weights(), R_max, R_min)

        # shape: (f, c)
        power_matrix = table.lookup(R).numpy()

        mean_power_train, Y_power_train = compute_mean_power(
            X_rate_data_train,
            power_matrix
        )

        power_results[f_sample].append(mean_power_train)

        summary_rows.append({
            "f_sample": f_sample,
            "seed": seed_value,
            "train_acc": train_acc,
            "test_acc": test_acc,
            "mean_power_train": mean_power_train
        })

        print(
            f"Seed {seed_value:2d} | "
            f"Train Acc = {train_acc:.4f} | "
            f"Test Acc = {test_acc:.4f} | "
            f"Mean Power = {mean_power_train:.6e}"
        )

    mean_over_seeds = np.mean(power_results[f_sample])
    std_over_seeds = np.std(power_results[f_sample])

    print(f"\nMean power over all seeds for f_sample={f_sample}: {mean_over_seeds:.6e}")
    print(f"Std  power over all seeds for f_sample={f_sample}: {std_over_seeds:.6e}")

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv("power_summary_by_seed.csv", index=False)

mean_df = (
    summary_df.groupby("f_sample", as_index=False)["mean_power_train"]
    .agg(["mean", "std"])
    .reset_index()
)
mean_df.to_csv("power_summary_by_fsample.csv", index=False)

print("\nSaved:")
print(" - power_summary_by_seed.csv")
print(" - power_summary_by_fsample.csv")

print("\nFinal mean power by f_sample:")
print(mean_df)


f_sample = 8
Seed  1 | Train Acc = 1.0000 | Test Acc = 0.9444 | Mean Power = 4.721276e+01
Seed  8 | Train Acc = 0.9919 | Test Acc = 0.9630 | Mean Power = 4.683281e+01
Seed 10 | Train Acc = 0.9919 | Test Acc = 0.9444 | Mean Power = 4.696721e+01
Seed 11 | Train Acc = 1.0000 | Test Acc = 0.9259 | Mean Power = 4.709030e+01
Seed 21 | Train Acc = 0.9919 | Test Acc = 0.9444 | Mean Power = 4.660658e+01
Seed 25 | Train Acc = 1.0000 | Test Acc = 0.9074 | Mean Power = 4.705730e+01
Seed 28 | Train Acc = 0.9919 | Test Acc = 0.9815 | Mean Power = 4.663093e+01
Seed 32 | Train Acc = 1.0000 | Test Acc = 0.9259 | Mean Power = 4.744178e+01
Seed 36 | Train Acc = 0.9919 | Test Acc = 0.9259 | Mean Power = 4.768656e+01

Mean power over all seeds for f_sample=8: 4.705847e+01
Std  power over all seeds for f_sample=8: 3.354100e-01

f_sample = 16
Seed  1 | Train Acc = 1.0000 | Test Acc = 0.9444 | Mean Power = 5.395916e+01
Seed  8 | Train Acc = 0.9919 | Test Acc = 0.9630 | Mean Power = 5.379601e+01
Seed 10 | Tra